# DGL Benchmark

Run this notebook in the **DGL environment**. It saves results to `comparison_outputs/` so `compare_results.ipynb` can compare them side-by-side with the PyG run.

Stages:
1. Graph construction — node/edge counts per type
2. Training — 1 pretrain epoch + 5 finetune epochs, save validation metrics
3. Disease-centric evaluation — per-disease AUROC on a sample of test diseases

In [ ]:
import sys, os, json, pickle
import torch
import numpy as np

REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd()))
DGL_ROOT    = os.path.join(REPO_ROOT, 'dgl_implementation')
OUT_DIR     = os.path.join(REPO_ROOT, 'comparison_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, DGL_ROOT)
import txgnn

print('txgnn from:', txgnn.__file__)
print('Output dir:', OUT_DIR)

In [ ]:
# ── Shared config (must match pyg_benchmark.ipynb exactly) ──
DATA_FOLDER   = os.path.join(REPO_ROOT, 'data')
SPLIT         = 'complex_disease'
SEED          = 42
DEVICE        = 'cuda:0' if torch.cuda.is_available() else 'cpu'
N_HID         = 100
N_INP         = 100
N_OUT         = 100
PROTO         = True
PROTO_NUM     = 5
ATTENTION     = False
SIM_MEASURE   = 'all_nodes_profile'
AGG_MEASURE   = 'rarity'
N_PRETRAIN    = 1
N_FINETUNE    = 5
BATCH_SIZE    = 1024
LR            = 1e-3
SAMPLE_DISEASES = 5

DD_ETYPES = [
    ('drug', 'contraindication', 'disease'),
    ('drug', 'indication', 'disease'),
    ('drug', 'off-label use', 'disease'),
    ('disease', 'rev_contraindication', 'drug'),
    ('disease', 'rev_indication', 'drug'),
    ('disease', 'rev_off-label use', 'drug'),
]

print('Device:', DEVICE)

---
## Stage 1 — Graph construction

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

data = txgnn.TxData(data_folder_path=DATA_FOLDER)
data.prepare_split(split=SPLIT, seed=SEED)
G = data.G

print('Graph loaded')
print('  ntypes:', G.ntypes)
print('  # canonical edge types:', len(G.canonical_etypes))

In [ ]:
graph_stats = {
    'node_counts': {ntype: G.number_of_nodes(ntype) for ntype in G.ntypes},
    'edge_counts': {str(et): G.number_of_edges(et) for et in G.canonical_etypes},
    'edge_indices': {},
}

for et in G.canonical_etypes:
    src, dst = G.edges(etype=et)
    idx = np.lexsort((dst.numpy(), src.numpy()))
    graph_stats['edge_indices'][str(et)] = {
        'src': src.numpy()[idx].tolist(),
        'dst': dst.numpy()[idx].tolist(),
    }

with open(os.path.join(OUT_DIR, 'dgl_graph_stats.json'), 'w') as f:
    json.dump(graph_stats, f)

print('Stage 1 saved: dgl_graph_stats.json')
print('  node types:', list(graph_stats['node_counts'].keys()))
print('  edge type count:', len(graph_stats['edge_counts']))

---
## Stage 2 — Training metrics

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

model = txgnn.TxGNN(data=data, device=DEVICE)
model.model_initialize(
    n_hid=N_HID, n_inp=N_INP, n_out=N_OUT,
    proto=PROTO, proto_num=PROTO_NUM,
    attention=ATTENTION,
    sim_measure=SIM_MEASURE,
    agg_measure=AGG_MEASURE,
)
print('Model initialized, params:', sum(p.numel() for p in model.model.parameters()))

In [ ]:
torch.manual_seed(0)
model.pretrain(
    n_epoch=N_PRETRAIN,
    learning_rate=LR,
    batch_size=BATCH_SIZE,
    train_print_per_n=9999,
)
print('Pretrain done')

In [ ]:
torch.manual_seed(0)
model.finetune(
    n_epoch=N_FINETUNE,
    learning_rate=LR,
    train_print_per_n=9999,
    valid_per_n=N_FINETUNE,
)
print('Finetune done')

In [ ]:
from txgnn.utils import evaluate_fb

(auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), loss = \
    evaluate_fb(model.best_model, model.g_valid_pos, model.g_valid_neg,
                data.G.to(DEVICE), DD_ETYPES, DEVICE)

metrics = {
    'macro_auroc': macro_auroc,
    'macro_auprc': macro_auprc,
    'micro_auroc': micro_auroc,
    'micro_auprc': micro_auprc,
    'loss': loss,
    'auroc_per_etype': {str(k): v for k, v in auroc_rel.items()},
    'auprc_per_etype': {str(k): v for k, v in auprc_rel.items()},
}

with open(os.path.join(OUT_DIR, 'dgl_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print('Stage 2 saved: dgl_metrics.json')
print(f'  Macro AUROC={macro_auroc:.4f}  Macro AUPRC={macro_auprc:.4f}  Loss={loss:.4f}')

In [ ]:
# Save model weights + node embeddings for optional forward-pass parity check in pyg_benchmark
torch.save(model.best_model.state_dict(), os.path.join(OUT_DIR, 'dgl_model_state.pt'))

node_emb = {ntype: data.G.nodes[ntype].data['inp'].detach().cpu() for ntype in data.G.ntypes}
with open(os.path.join(OUT_DIR, 'dgl_node_emb.pkl'), 'wb') as f:
    pickle.dump(node_emb, f)

print('Model state + node embeddings saved')

---
## Stage 3 — Disease-centric evaluation

In [ ]:
evaluator = txgnn.TxEval(model=model, data=data)
disease_ids = evaluator.retrieve_disease_idxs_test_set('indication')[:SAMPLE_DISEASES]
print('Evaluating diseases:', disease_ids.tolist())

result = evaluator.eval_disease_centric(
    disease_idxs=disease_ids.tolist(),
    relation='indication',
    return_raw=True,
    show_plot=False,
    verbose=False,
    simulate_random=False,
)

disease_auroc = {str(k): float(v) for k, v in result['result']['AUROC'].items()}

with open(os.path.join(OUT_DIR, 'dgl_disease_auroc.json'), 'w') as f:
    json.dump({'disease_ids': disease_ids.tolist(), 'auroc': disease_auroc}, f, indent=2)

print('Stage 3 saved: dgl_disease_auroc.json')
for did, auc in disease_auroc.items():
    print(f'  disease {did}: AUROC={auc:.4f}')

---
All outputs saved to `comparison_outputs/`. Run `pyg_benchmark.ipynb` next, then open `compare_results.ipynb` to see the side-by-side comparison.